In [59]:
import pandas as pd
import numpy as np

np.random.seed(101)

channels = ["Google ads","instagram","facebook ads","linkedin ads"]
dates_pool = ["2025/01/01", "01/02/2025", "2025.01.03", "2025/01/04" , "05-jan-2025"]
raw_data = []
for i in range(2000):
    channel = np.random.choice(channels)
    choice = np.random.randint(100,1500)
    clicks = np.random.randint(10,30)
    impressions = np.random.randint(100,1000)

    base_conv_rate = 0.03 if channel != "instagram" else 0.052
    conversions = int(clicks * base_conv_rate + np.random.randint(0,5))

    currency = np.random.choice (["USD","EUR","INR"])
    row_spend = np.random.randint(200,3000)

    dirty_date = np.random.choice(dates_pool)
    revenue = conversions * np.random.randint(40,60)

    raw_data.append ([dirty_date,channel,currency,row_spend,impressions,clicks,conversions,revenue])

df = pd.DataFrame(raw_data,columns= ["date", "channel","currency","row_spend","impressions","clicks","conversions","revenue"])

df["clean_data"] = pd.to_datetime (df["date"], errors = "coerce").dt.strftime("%Y-%m-%d")

def convert_to_usd(row):
    if row ["currency"] == "EUR":
        return row ["row_spend"]/0.92
    elif row ["currency"] == "INR":
         return row ["row_spend"]/83.0
         return row ["row_spend"]

df["adspend_usd"] = df.apply(convert_to_usd, axis=1)

campaign_summary = df.groupby ("channel").agg(
    Total_spend = ("row_spend","sum"),
    Total_impressions = ("impressions","sum"),
    Total_clicks = ("clicks","sum"),
    Total_conversion = ("conversions","sum"),
    Total_revenue = ("revenue","sum")
).reset_index()

campaign_summary ["CTR_%"] = (campaign_summary["Total_clicks"]/campaign_summary["Total_impressions"])*100
campaign_summary ["cpc_usd"] = (campaign_summary["Total_spend"]/campaign_summary["Total_clicks"])
campaign_summary ["conversion_rate_%"] = (campaign_summary["Total_conversion"]/campaign_summary["Total_clicks"])*100
campaign_summary ["roars"] = campaign_summary["Total_revenue"]/campaign_summary["Total_spend"]

print("=== multi channel performance evaluation ====")
print(campaign_summary[["channel", "Total_spend", "cpc_usd", "conversion_rate_%","roars"]].round(2))

insta_conv = campaign_summary.loc [campaign_summary ["channel"] == "instagram","conversion_rate_%"].sum()
facebook_conv = campaign_summary.loc[campaign_summary ["channel"]== "facebook_ads","conversion_rate_%"].sum()

pct_lift = ((insta_conv-facebook_conv)/facebook_conv*100) if facebook_conv !=0 else 0

print("\n=== system anomaly insight ===")
print(f"instagram generated a verified {pct_lift: .2f}% relative lift in customer conversion rate over facebook Ads.")

df.to_csv ("cleaned_marketing_data.csv",index=False)

=== multi channel performance evaluation ====
        channel  Total_spend  cpc_usd  conversion_rate_%  roars
0    Google ads       795698    80.70              10.35   0.06
1  facebook ads       783474    81.21              10.69   0.07
2     instagram       812138    81.00              13.29   0.08
3  linkedin ads       814812    84.43              10.53   0.06

=== system anomaly insight ===
instagram generated a verified  0.00% relative lift in customer conversion rate over facebook Ads.
